# LAB 4 (opcional) — Coleta real de posts do X via API

**Disciplina** Engenharia de Dados para IA

**Aula** 4 — Arquiteturas Modernas de Dados

**Complemento da** Atividade 1.3 do LAB 4 (Relacional vs. Não-relacional)

> **Opcional e fora da avaliação.** O LAB 4 já traz 10 posts fictícios. Use este notebook só se quiser repetir a Atividade 1.3 com **posts reais**, usando a sua própria conta de desenvolvedor do X.

> **Atenção: tem custo.** A API do X é cobrada por uso (*pay-per-use*), cerca de **US$ 0,005 por post lido** em out/2026. Com os parâmetros padrão (10 posts, 1 página), a execução custa ~US$ 0,05. Confira o preço vigente no Developer Console antes de rodar.

**O que este notebook faz:**

1. Busca posts recentes (últimos 7 dias) com `GET /2/tweets/search/recent`.
2. Grava a resposta **bruta** em arquivo JSONL num Volume (zona de pouso).
3. Carrega a **Bronze** de forma idempotente (`MERGE` por `post_id`). Rodar duas vezes não duplica posts.
4. Gera a **Silver** com a mesma lógica da Atividade 1.3 e cruza com `fato_entregas`.

**Pré-requisitos:** seed do LAB 2/3 carregado (`lodlog_dw`) e Atividade 1.3 do LAB 4 já feita.

## Passo 1 — Conta de desenvolvedor e Bearer Token

1. Acesse **developer.x.com** com uma conta do X e abra o **Developer Console**.
2. Crie um **App** e aceite os termos de uso de desenvolvedor.
3. Em **Billing**, adicione créditos. Defina um valor baixo (ex.: US$ 5), que sobra para a atividade.
4. Em **Keys and tokens** do App, gere o **Bearer Token** (autenticação *app-only*, suficiente para leitura de posts públicos).
5. Copie o token **uma única vez** e guarde-o como segredo (Passo 2).

> **Nunca** cole o token numa célula do notebook, num commit ou no relatório. Quem tiver o token gasta os seus créditos.

## Passo 2 — Guardar o token como segredo

**No Databricks (recomendado):** use o *secret scope*. No terminal do seu computador, com a [Databricks CLI](https://docs.databricks.com/dev-tools/cli/) autenticada no workspace:

```bash
databricks secrets create-scope lodlog
databricks secrets put-secret lodlog x_bearer_token   # cole o token quando pedir
```

**Fora do Databricks** (Colab, Jupyter local): defina a variável de ambiente `X_BEARER_TOKEN`. Se ela não existir, a célula abaixo pede o token sem exibi-lo.

In [ ]:
import os
from getpass import getpass

EM_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

def ler_token():
    if EM_DATABRICKS:
        try:
            return dbutils.secrets.get(scope="lodlog", key="x_bearer_token")
        except Exception as e:
            print(f"Secret lodlog/x_bearer_token não encontrado ({type(e).__name__}).")
    return os.environ.get("X_BEARER_TOKEN") or getpass("Bearer Token do X: ")

X_BEARER_TOKEN = ler_token()
print("Token carregado:", "sim" if X_BEARER_TOKEN else "NÃO")

## Passo 3 — O ambiente alcança a API do X?

No **Databricks Free Edition**, a computação serverless só acessa a internet para uma lista restrita de domínios confiáveis, e `api.x.com` pode estar bloqueado. Este teste não gasta créditos, porque faz uma chamada sem token:

* **HTTP 401** → a API é alcançável. Siga para o Passo 4 aqui mesmo.
* **Erro de conexão/DNS** → bloqueado. Use o **Plano B**: rode os Passos 2, 4 e 5 fora do Databricks (Google Colab ou Python local), depois faça upload do arquivo `.jsonl` gerado para o Volume (Passo 6).

In [ ]:
import requests

try:
    r = requests.get("https://api.x.com/2/tweets/search/recent", timeout=10)
    API_ACESSIVEL = True
    print(f"HTTP {r.status_code}: API alcançável (401 é o esperado sem token).")
except requests.exceptions.RequestException as e:
    API_ACESSIVEL = False
    print(f"Sem acesso à API ({type(e).__name__}). Use o Plano B.")

## Passo 4 — Parâmetros da busca e custo estimado

A query usa os [operadores de busca do X](https://docs.x.com/x-api/posts/search/integrate/build-a-query). Os filtros reduzem o custo, porque só posts devolvidos são cobrados:

* `lang:pt`: só português.
* `-is:retweet`: retweets são duplicatas (na Atividade 1.3 eles eram descartados na Silver; aqui nem chegam a ser pagos).
* Palavras de evento **E** cidades dos CDs da LODLog.

A API devolve de 10 a 100 posts por página. Comece com o padrão e só aumente se precisar.

In [ ]:
QUERY = (
    "(bloqueio OR bloqueada OR greve OR protesto OR acidente OR tombamento "
    "OR congestionamento OR interditada) "
    '(Campinas OR "São Paulo" OR "Belo Horizonte" OR Curitiba OR "Porto Alegre") '
    "lang:pt -is:retweet"
)
MAX_RESULTS = 10      # posts por página (10 a 100)
MAX_PAGINAS = 1       # páginas no máximo
PRECO_POR_POST = 0.005  # US$ — confira no Developer Console

print(QUERY)
print(f"Custo máximo estimado: {MAX_RESULTS * MAX_PAGINAS} posts × US$ {PRECO_POR_POST} "
      f"= US$ {MAX_RESULTS * MAX_PAGINAS * PRECO_POR_POST:.2f}")

## Passo 5 — Coleta (chamada à API)

Cada página da API traz uma lista de posts em `data` e, separadamente, os usuários e locais referenciados em `includes`. Para manter **o mesmo formato da Bronze da Atividade 1.3**, a função `desmontar_pagina` gera **um registro por post**, cada um com o seu `includes`.

O arquivo é gravado **exatamente como veio da API** (fora essa separação por post). Nada é filtrado ou renomeado aqui. Essa é a zona de pouso.

In [ ]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

URL_BUSCA = "https://api.x.com/2/tweets/search/recent"
CAMPOS = {
    "tweet.fields": "created_at,author_id,lang,public_metrics,entities,geo,referenced_tweets",
    "expansions": "author_id,geo.place_id",
    "user.fields": "username",
    "place.fields": "full_name",
}

def desmontar_pagina(resposta):
    includes = resposta.get("includes", {})
    usuarios = {u["id"]: u for u in includes.get("users", [])}
    locais = {p["id"]: p for p in includes.get("places", [])}
    registros = []
    for post in resposta.get("data", []):
        inc = {}
        if post.get("author_id") in usuarios:
            inc["users"] = [usuarios[post["author_id"]]]
        place_id = post.get("geo", {}).get("place_id")
        if place_id in locais:
            inc["places"] = [locais[place_id]]
        registros.append({"data": post, "includes": inc})
    return registros

def coletar(query, max_results, max_paginas, token):
    headers = {"Authorization": f"Bearer {token}"}
    params = {**CAMPOS, "query": query, "max_results": max_results}
    registros, paginas = [], 0
    while paginas < max_paginas:
        r = requests.get(URL_BUSCA, headers=headers, params=params, timeout=30)
        if r.status_code == 429:
            reset = int(r.headers.get("x-rate-limit-reset", time.time() + 60))
            espera = min(max(reset - time.time(), 1), 900)
            print(f"Limite de requisições atingido; aguardando {espera:.0f}s...")
            time.sleep(espera)
            continue
        if r.status_code != 200:
            raise RuntimeError(f"HTTP {r.status_code}: {r.text[:500]}")
        resposta = r.json()
        paginas += 1
        registros += desmontar_pagina(resposta)
        meta = resposta.get("meta", {})
        print(f"Página {paginas}: {meta.get('result_count', 0)} posts")
        if "next_token" not in meta:
            break
        params["next_token"] = meta["next_token"]
    return registros

registros = coletar(QUERY, MAX_RESULTS, MAX_PAGINAS, X_BEARER_TOKEN)
print(f"Total coletado: {len(registros)} posts "
      f"(custo aproximado: US$ {len(registros) * PRECO_POR_POST:.2f})")

In [ ]:
# Zona de pouso: Volume no Databricks; pasta local no Plano B
if EM_DATABRICKS:
    spark.sql("CREATE SCHEMA IF NOT EXISTS lodlog_lake")
    spark.sql("CREATE VOLUME IF NOT EXISTS lodlog_lake.raw")
    catalogo = spark.catalog.currentCatalog()
    PASTA_POUSO = Path(f"/Volumes/{catalogo}/lodlog_lake/raw/x_posts")
else:
    PASTA_POUSO = Path("x_posts")
PASTA_POUSO.mkdir(parents=True, exist_ok=True)

carimbo = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
arquivo = PASTA_POUSO / f"x_posts_{carimbo}.jsonl"
with open(arquivo, "w", encoding="utf-8") as f:
    for reg in registros:
        f.write(json.dumps(reg, ensure_ascii=False) + "\n")
print(f"{len(registros)} posts gravados em {arquivo}")

## Passo 6 — Bronze (no Databricks)

**Plano B:** se a coleta rodou fora do Databricks, crie o Volume com a célula SQL abaixo e faça upload dos arquivos `x_posts_*.jsonl`. No menu **Catalog**, abra `workspace` → `lodlog_lake` → `raw`, crie a pasta `x_posts` e use **Upload to this volume**.

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS lodlog_lake;
CREATE VOLUME IF NOT EXISTS lodlog_lake.raw;

CREATE TABLE IF NOT EXISTS lodlog_lake.x_posts_bronze_api (
  post_id        STRING,
  payload        STRING,
  arquivo_origem STRING,
  ingerido_em    TIMESTAMP
) USING DELTA;

A carga lê **todos** os arquivos da zona de pouso e usa `MERGE` por `post_id`: posts já carregados são ignorados, só os novos entram. A carga é **idempotente**, ou seja, pode ser repetida sem duplicar dados. O mesmo post também pode vir em duas coletas seguidas, e a deduplicação na leitura (`dropDuplicates`) cuida disso.

In [ ]:
from pyspark.sql import functions as F

catalogo = spark.catalog.currentCatalog()
novos = (spark.read.text(f"/Volumes/{catalogo}/lodlog_lake/raw/x_posts/*.jsonl")
         .select(F.get_json_object("value", "$.data.id").alias("post_id"),
                 F.col("value").alias("payload"),
                 F.col("_metadata.file_name").alias("arquivo_origem"),
                 F.current_timestamp().alias("ingerido_em"))
         .where("post_id IS NOT NULL")
         .dropDuplicates(["post_id"]))
novos.createOrReplaceTempView("x_posts_novos")

spark.sql('''
MERGE INTO lodlog_lake.x_posts_bronze_api b
USING x_posts_novos n
ON b.post_id = n.post_id
WHEN NOT MATCHED THEN INSERT *
''').show()

spark.sql("SELECT COUNT(*) AS posts_na_bronze FROM lodlog_lake.x_posts_bronze_api").show()

**Explorar o JSON real.** Que fração dos posts reais tem localização (`places`)? E hashtags? Compare com os 10 posts fictícios da Atividade 1.3.

In [ ]:
%sql
SELECT COUNT(*)                                                                AS posts,
       COUNT(get_json_object(payload, '$.includes.places[0].full_name'))      AS com_local,
       COUNT(get_json_object(payload, '$.data.entities.hashtags'))            AS com_hashtags,
       COUNT(get_json_object(payload, '$.data.referenced_tweets[0].type'))    AS com_referencia,
       MIN(get_json_object(payload, '$.data.created_at'))                     AS mais_antigo,
       MAX(get_json_object(payload, '$.data.created_at'))                     AS mais_recente
FROM lodlog_lake.x_posts_bronze_api;

In [ ]:
%sql
SELECT get_json_object(payload, '$.data.id')                        AS post_id,
       get_json_object(payload, '$.data.created_at')                AS criado_em_utc,
       get_json_object(payload, '$.includes.places[0].full_name')   AS local,
       get_json_object(payload, '$.data.referenced_tweets[0].type') AS referencia,
       get_json_object(payload, '$.data.text')                      AS texto
FROM lodlog_lake.x_posts_bronze_api
ORDER BY criado_em_utc DESC;

## Passo 7 — Silver: mesma regra da Atividade 1.3

A regra de classificação é **idêntica** à da Atividade 1.3, para vocês compararem como ela se sai com dados reais. Quotes e respostas (`quoted`, `replied_to`) são mantidos, e o autor **não** passa para a Silver (minimização, LGPD).

In [ ]:
%sql
CREATE OR REPLACE TABLE lodlog_lake.x_eventos_silver_api USING DELTA AS
WITH posts AS (
  SELECT post_id,
         to_date(from_utc_timestamp(
           CAST(get_json_object(payload, '$.data.created_at') AS TIMESTAMP),
           'America/Sao_Paulo')) AS data_evento,
         lower(get_json_object(payload, '$.data.text')) AS texto,
         split(get_json_object(payload, '$.includes.places[0].full_name'), ',')[0] AS cidade_place,
         get_json_object(payload, '$.data.referenced_tweets[0].type') AS referencia
  FROM lodlog_lake.x_posts_bronze_api
)
SELECT post_id,
       data_evento,
       COALESCE(cidade_place,
                CASE WHEN texto LIKE '%campinas%'       THEN 'Campinas'
                     WHEN texto LIKE '%são paulo%'      THEN 'São Paulo'
                     WHEN texto LIKE '%belo horizonte%' THEN 'Belo Horizonte'
                     WHEN texto LIKE '%curitiba%'       THEN 'Curitiba'
                     WHEN texto LIKE '%porto alegre%'   THEN 'Porto Alegre'
                END) AS cidade,
       CASE WHEN texto RLIKE 'bloque|protesto|manifesta'       THEN 'Bloqueio/Protesto'
            WHEN texto RLIKE 'greve|paralisa'                  THEN 'Greve'
            WHEN texto RLIKE 'acidente|tombamento|colis'       THEN 'Acidente'
            WHEN texto RLIKE 'congestionamento|lentid|parada'  THEN 'Congestionamento'
            ELSE 'Irrelevante'
       END AS tipo_evento
FROM posts
WHERE referencia IS NULL OR referencia <> 'retweeted';

SELECT tipo_evento, cidade, COUNT(*) AS posts
FROM lodlog_lake.x_eventos_silver_api
GROUP BY tipo_evento, cidade
ORDER BY posts DESC;

## Passo 8 — Cruzar com as entregas da LODLog

As entregas do seed vão só até **30/09/2026**, e a busca recente cobre os **últimos 7 dias**. Por isso `entregas_no_dia` tende a sair **0** para os posts reais. A consulta ainda mostra, para cada evento, qual CD seria afetado e a taxa de atraso normal dele. Em produção, os dois lados seriam atuais.

In [ ]:
%sql
WITH eventos AS (
  SELECT data_evento, cidade, concat_ws(', ', collect_set(tipo_evento)) AS eventos,
         COUNT(*) AS posts
  FROM lodlog_lake.x_eventos_silver_api
  WHERE tipo_evento <> 'Irrelevante' AND cidade IS NOT NULL
  GROUP BY data_evento, cidade
),
normal_cd AS (
  SELECT sk_cd_origem, ROUND(100 * AVG(indicador_atraso), 1) AS pct_atraso_normal
  FROM lodlog_dw.fato_entregas
  GROUP BY sk_cd_origem
)
SELECT e.data_evento, e.cidade, e.eventos, e.posts, cd.codigo AS cd,
       COUNT(fe.sk_entrega)                    AS entregas_no_dia,
       ROUND(100 * AVG(fe.indicador_atraso), 1) AS pct_atraso_no_dia,
       n.pct_atraso_normal
FROM eventos e
JOIN lodlog_dw.dim_centro_distribuicao cd ON cd.municipio = e.cidade
JOIN normal_cd n                          ON n.sk_cd_origem = cd.sk_cd
LEFT JOIN lodlog_dw.fato_entregas fe      ON fe.sk_cd_origem = cd.sk_cd
                                         AND fe.data_entrega = e.data_evento
GROUP BY e.data_evento, e.cidade, e.eventos, e.posts, cd.codigo, n.pct_atraso_normal
ORDER BY e.data_evento;

## Reflexão (opcional, para discussão)

1. Que fração dos posts reais trouxe localização (`places`)? O que isso muda na estratégia de identificar a cidade, em comparação com os posts fictícios?
2. Leia 10 posts classificados. Quantos a regra de palavras-chave acertou? Liste falsos positivos (ex.: "greve" no sentido figurado) e falsos negativos.
3. Quanto custou a coleta? Estime o custo **mensal** de rodar essa busca a cada 15 minutos com 100 posts por página. Vale a pena? Que filtros reduziriam o custo?
4. A Bronze guarda `author_id` e `username` de pessoas reais. Por quanto tempo esses dados podem ficar lá (LGPD e termos de uso do X)? O que acontece se o autor apagar o post?
5. Por que a carga com `MERGE` por `post_id` é importante se a coleta rodar de forma agendada (ex.: a cada hora)?

## Limpeza

Ao terminar, apague os dados reais coletados. Eles contêm dados pessoais, e os termos do X exigem respeitar posts apagados pelos autores. Mude `APAGAR` para `True` e rode a célula.

In [ ]:
APAGAR = False

if APAGAR:
    catalogo = spark.catalog.currentCatalog()
    spark.sql("DROP TABLE IF EXISTS lodlog_lake.x_eventos_silver_api")
    spark.sql("DROP TABLE IF EXISTS lodlog_lake.x_posts_bronze_api")
    dbutils.fs.rm(f"/Volumes/{catalogo}/lodlog_lake/raw/x_posts", recurse=True)
    print("Dados do X removidos.")